# Project 2 — HR Policy Bot | LoRA Fine-Tuning
### Shruti Agrawal | Accenture LLM Developer Interview Prep

---

**What we're building:**
Fine-tuning TinyLlama to answer employee HR policy questions — leave policies, reimbursements, onboarding, benefits, conduct rules.
This is a real enterprise use case Accenture builds for clients.

**What's different from Project 1:**
- Minimal comments — you should recognise most of this now
- More training examples (20) — better output quality
- Slightly tuned hyperparameters — watch how loss behaves differently
- You'll see domain adaptation in action — the model shifts from general to HR-specific

**Before running:** Runtime → Change Runtime Type → T4 GPU → Save

---

## Cell 1 — Install libraries

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

## Cell 2 — Imports + GPU check

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch

print("GPU available:", "YES —", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO — go to Runtime > Change Runtime Type > T4 GPU")

## Cell 3 — Training data

20 HR policy instruction-response pairs.
Notice the domain is consistent — formal, policy-accurate, structured responses.
This consistency in training data is what teaches the model the right tone and format.
This is your annotation and guideline work from NVIDIA — applied here as code.

In [ ]:
training_examples = [
    {
        "instruction": "How many casual leaves am I entitled to per year?",
        "response": "As per company policy, all full-time employees are entitled to 12 casual leaves per calendar year. These cannot be carried forward to the next year and are credited at the start of each January. Probationary employees receive casual leaves on a pro-rata basis from their date of joining."
    },
    {
        "instruction": "What is the process to apply for maternity leave?",
        "response": "Maternity leave must be applied at least 30 days before the expected date of delivery. You are entitled to 26 weeks of paid maternity leave as per the Maternity Benefit Act. Please submit Form ML-01 to your HR Business Partner along with the doctor's certificate confirming the expected due date."
    },
    {
        "instruction": "Can I work from home permanently?",
        "response": "Permanent work from home arrangements require approval from your Department Head and HR. The decision is based on the nature of your role, team requirements, and performance history. You may submit a formal request through the HR portal under the 'Flexible Work Arrangement' section. All approvals are reviewed quarterly."
    },
    {
        "instruction": "How do I claim travel reimbursement for a client visit?",
        "response": "To claim travel reimbursement, submit your expenses within 7 working days of the trip via the expense management portal. Attach original receipts for all claims above Rs. 500. Approved modes of travel for client visits include flights (economy class), trains, and cabs booked through the company travel desk. Claims submitted after 7 days may not be processed."
    },
    {
        "instruction": "What happens to my earned leaves if I don't use them?",
        "response": "Earned leaves can be accumulated up to a maximum of 45 days. Leaves beyond this cap will lapse at the end of the financial year. Employees may also opt for leave encashment for up to 15 earned leaves per year, subject to management approval. This can be requested during the annual leave review cycle in March."
    },
    {
        "instruction": "What is the notice period for resignation?",
        "response": "The standard notice period is 60 days for all confirmed employees. During your probation period, the notice period is 30 days. Notice period buyout is available subject to business requirements and manager approval. Your relieving letter and full and final settlement will be processed within 30 days of your last working day."
    },
    {
        "instruction": "How do I update my bank account details for salary?",
        "response": "Bank account updates must be submitted at least 10 working days before the salary processing date to ensure timely credit. Log in to the HR self-service portal, navigate to 'My Profile', and select 'Bank Details'. Attach a cancelled cheque or bank statement as proof. Changes are verified by payroll before activation."
    },
    {
        "instruction": "Am I eligible for a performance bonus this year?",
        "response": "Performance bonuses are awarded to employees who have completed at least 6 months of service in the appraisal cycle and have received a performance rating of 'Meets Expectations' or above. Bonus amounts are determined by your band, rating, and the company's overall financial performance. Bonus letters are shared by HR in April each year."
    },
    {
        "instruction": "What is the policy on internet usage at work?",
        "response": "Company internet resources are to be used for business purposes only. Accessing non-work related streaming platforms, gaming sites, or social media for personal use during work hours is not permitted on company networks. All internet activity on company devices is monitored. Violation of this policy may result in disciplinary action as per the Code of Conduct."
    },
    {
        "instruction": "How do I report a workplace harassment complaint?",
        "response": "All harassment complaints are handled with strict confidentiality under the Prevention of Sexual Harassment (POSH) policy. You may report an incident by emailing the Internal Complaints Committee at posh@company.com or by speaking directly to your HR Business Partner. A formal inquiry will be initiated within 7 working days of the complaint being registered. Retaliation against complainants is strictly prohibited."
    },
    {
        "instruction": "What benefits are included in my health insurance?",
        "response": "The company's group health insurance covers you, your spouse, and up to two dependent children. Coverage includes hospitalisation, daycare procedures, pre and post-hospitalisation expenses up to 30 and 60 days respectively, and maternity benefits. The annual sum insured is Rs. 5 lakhs per family. You may opt for a top-up plan at a subsidised premium during the annual enrolment window."
    },
    {
        "instruction": "Can I take a sabbatical leave?",
        "response": "Sabbatical leave is available to employees with more than 5 years of continuous service. You may apply for an unpaid sabbatical of up to 6 months for purposes such as higher education, personal development, or caregiving. Approval is subject to business continuity requirements and must be applied for at least 60 days in advance through the HR portal."
    },
    {
        "instruction": "What is the process for internal job transfers?",
        "response": "Internal transfers are encouraged and can be initiated after completing 18 months in your current role. Browse open positions on the internal careers portal and apply directly. Inform your current manager before applying as a professional courtesy, though their approval is not mandatory. HR will coordinate between both teams to manage the transition timeline, typically 30 to 45 days."
    },
    {
        "instruction": "How is my provident fund contribution calculated?",
        "response": "Your Provident Fund contribution is 12% of your basic salary, matched equally by the employer. Both contributions are deposited monthly with the Employee Provident Fund Organisation (EPFO). You can view your PF balance and download your passbook by logging into the EPFO member portal using your UAN number. Your UAN is available in your salary slip and the HR portal."
    },
    {
        "instruction": "What is the dress code policy?",
        "response": "The company follows a business casual dress code from Monday to Thursday. Fridays are designated casual days where smart casuals are acceptable. When attending client meetings or office events, formal business attire is expected regardless of the day. Clothing with offensive graphics, slogans, or torn clothing is not permitted at any time on company premises."
    },
    {
        "instruction": "How do I apply for a study leave?",
        "response": "Study leave may be granted for courses or examinations directly relevant to your current role or career development at the company. You are eligible for up to 5 paid study leave days per year. Submit your application through the HR portal at least 14 days before the required date, attaching proof of enrollment or examination schedule. Approval rests with your line manager and HR."
    },
    {
        "instruction": "What happens if I am absent without approval?",
        "response": "Unapproved absences are marked as Loss of Pay (LOP) for the days concerned. If an employee is absent without communication for more than 5 consecutive working days, it may be treated as voluntary abandonment of employment, which can lead to termination. It is important to always inform your manager and apply for leave in advance or on the same day in case of emergencies."
    },
    {
        "instruction": "Is there a referral bonus for recommending a candidate?",
        "response": "Yes, the employee referral program offers a bonus of Rs. 25,000 for successful referrals at junior and mid levels, and Rs. 50,000 for senior level positions. The referred candidate must complete 90 days of employment before the bonus is paid out. Submit referrals through the HR portal under 'Refer a Friend'. Self-referrals and referrals of immediate family members are not eligible."
    },
    {
        "instruction": "Can I opt out of the National Pension Scheme?",
        "response": "Enrollment in the National Pension Scheme (NPS) is optional for employees below the age of 60. If you wish to opt in or out, you may do so during the annual benefits window in April. Changes made outside this window are not processed until the next cycle. Contact your HR Business Partner or the payroll team for guidance on NPS contribution rates and tax benefits."
    },
    {
        "instruction": "What is the policy for using company laptops for personal use?",
        "response": "Company-issued laptops are provided for business use only. Limited personal use is tolerated provided it does not interfere with work, consume excessive bandwidth, or involve downloading unauthorised software. Installation of any software not approved by the IT department is strictly prohibited. Company laptops must not be used for illegal activities. All data on company devices is subject to monitoring and audit."
    },
]

def format_example(ex):
    return {"text": f"""<|system|>
You are an HR policy assistant for a large enterprise. Answer employee questions accurately, professionally, and based strictly on company policy.</s>
<|user|>
{ex['instruction']}</s>
<|assistant|>
{ex['response']}</s>"""}

dataset = Dataset.from_list([format_example(ex) for ex in training_examples])

print(f"Training examples loaded: {len(dataset)}")
print("-" * 50)
print(dataset[0]["text"])

## Cell 4 — Load model with 4-bit quantization

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading TinyLlama...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model = prepare_model_for_kbit_training(model)

print("Model ready on:", next(model.parameters()).device)

## Cell 5 — Apply LoRA

Same concept as Project 1 — but notice r=16 this time instead of 8.
Higher rank = LoRA matrices can capture more complex patterns.
Useful here because HR policy language is more structured and precise than customer support.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 6 — Train

Watch the loss column as it prints.
It should start around 1.5–2.5 and drop steadily.
If it drops below 0.8 the model is learning well.
If it stays flat or goes up — that would mean something is wrong.

In [ ]:
training_args = SFTConfig(
    output_dir="./tinyllama-hr-bot",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    bf16=True,
    logging_steps=5,
    save_steps=100,
    max_length=512,
    dataset_text_field="text",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

print("Training started — watch the loss go down...")
print("-" * 50)
trainer.train()
print("-" * 50)
print("Training complete!")

## Cell 7 — Save the LoRA adapter

In [ ]:
trainer.model.save_pretrained("./tinyllama-hr-bot-final")
tokenizer.save_pretrained("./tinyllama-hr-bot-final")
print("Adapter saved to: ./tinyllama-hr-bot-final")

## Cell 8 — Test the HR Bot

We test on questions the model has NOT seen during training.
If responses are coherent and policy-appropriate — the model has generalised well.
This is what 'evaluating model performance' means in the JD.

In [ ]:
def get_hr_response(question, max_new_tokens=200):
    prompt = f"""<|system|>
You are an HR policy assistant for a large enterprise. Answer employee questions accurately, professionally, and based strictly on company policy.</s>
<|user|>
{question}</s>
<|assistant|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.6,
            pad_token_id=tokenizer.eos_token_id
        )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|assistant|>" in full_response:
        return full_response.split("<|assistant|>")[-1].strip()
    return full_response.strip()


# These are unseen questions — not in our training data
test_questions = [
    "How many sick leaves do I get in a year?",
    "What is the process to apply for paternity leave?",
    "Can I use my earned leaves during notice period?",
]

print("HR Bot Responses on Unseen Questions")
print("=" * 60)
for q in test_questions:
    print(f"\nEmployee: {q}")
    print(f"HR Bot: {get_hr_response(q)}")
    print("-" * 60)

## Cell 9 — Ask it anything

In [ ]:
your_question = "What is the policy for working on public holidays?"

print(f"Employee: {your_question}")
print(f"\nHR Bot: {get_hr_response(your_question)}")

---
## What you can say in your interview about this project

*"For my second fine-tuning project I built an HR Policy Bot using TinyLlama 1.1B with QLoRA.
I designed 20 domain-specific instruction-response pairs covering leave policies, reimbursements,
conduct rules, and benefits — formatted in ChatML template for supervised fine-tuning.
I increased the LoRA rank from 8 to 16 compared to my first project to capture more complex
policy language patterns. I trained for 10 epochs and monitored loss to confirm the model
was learning, then evaluated on held-out HR questions it had not seen during training.
This is the kind of domain adaptation pipeline I understand Accenture builds for enterprise clients."*

---
## Key differences from Project 1 — know these for interviews:

| | Project 1 | Project 2 |
|---|---|---|
| Domain | Customer Support | HR Policy |
| Training examples | 10 | 20 |
| LoRA rank (r) | 8 | 16 |
| lora_alpha | 16 | 32 |
| Epochs | 3 | 10 |
| Temperature | 0.7 | 0.6 |
| Why lower temp? | More creative replies ok | Policy answers need precision |